# Speculative Prompt Caching

이 cookbook은 사용자가 아직 질문을 작성하는 동안 cache를 예열해 time-to-first-token(TTFT)을 줄이는 패턴인 "Speculative Prompt Caching"을 보여 줍니다.

**Speculative Caching 미사용:**
1. 사용자가 질문 입력(3초)
2. 사용자가 질문 제출
3. API가 context를 cache에 로드하고 응답도 생성

**Speculative Caching 사용:**
1. 사용자가 입력 시작(cache 예열 즉시 시작)
2. 사용자가 계속 입력(cache 예열은 백그라운드에서 완료)
3. 사용자가 질문 제출
4. API가 예열된 cache로 응답 생성

## 설정

먼저 필요한 package를 설치합니다.

In [82]:
%pip install anthropic httpx --quiet

Note: you may need to restart the kernel to use updated packages.


In [83]:
import asyncio
import copy
import datetime
import time

import httpx
from anthropic import AsyncAnthropic

# 설정 상수
MODEL = "claude-sonnet-4-6"
SQLITE_SOURCES = {
    "btree.h": "https://sqlite.org/src/raw/18e5e7b2124c23426a283523e5f31a4bff029131b795bb82391f9d2f3136fc50?at=btree.h",
    "btree.c": "https://sqlite.org/src/raw/63ca6b647342e8cef643863cd0962a542f133e1069460725ba4461dcda92b03c?at=btree.c",
}
DEFAULT_CLIENT_ARGS = {
    "system": "You are an expert systems programmer helping analyze database internals.",
    "max_tokens": 4096,
    "temperature": 0,
}

## Helper 함수

대규모 context를 다운로드하고 message를 준비하는 함수를 설정합니다.

In [84]:
async def get_sqlite_sources() -> dict[str, str]:
    print("Downloading SQLite source files...")

    source_files = {}
    start_time = time.time()

    async with httpx.AsyncClient(timeout=30.0) as client:
        tasks = []

        async def download_file(filename: str, url: str) -> tuple[str, str]:
            response = await client.get(url, follow_redirects=True)
            response.raise_for_status()
            print(f"Successfully downloaded {filename}")
            return filename, response.text

        for filename, url in SQLITE_SOURCES.items():
            tasks.append(download_file(filename, url))

        results = await asyncio.gather(*tasks)
        source_files = dict(results)

    duration = time.time() - start_time
    print(f"Downloaded {len(source_files)} files in {duration:.2f} seconds")
    return source_files


async def create_initial_message():
    sources = await get_sqlite_sources()
    # source code를 context로 사용하는 초기 message 준비.
    # 실행 간 cache 공유를 막기 위해 Timestamp 포함.
    initial_message = {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"""
Current time: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Source to Analyze:

btree.h:
```c
{sources["btree.h"]}
```

btree.c:
```c
{sources["btree.c"]}
```""",
                "cache_control": {"type": "ephemeral"},
            }
        ],
    }
    return initial_message


async def sample_one_token(client: AsyncAnthropic, messages: list):
    """cache 예열을 위해 single-token 요청 전송"""
    args = copy.deepcopy(DEFAULT_CLIENT_ARGS)
    args["max_tokens"] = 1
    await client.messages.create(
        messages=messages,
        model=MODEL,
        **args,
    )


def print_query_statistics(response, query_type: str) -> None:
    print(f"\n{query_type} query statistics:")
    print(f"\tInput tokens: {response.usage.input_tokens}")
    print(f"\tOutput tokens: {response.usage.output_tokens}")
    print(f"\tCache read input tokens: {getattr(response.usage, 'cache_read_input_tokens', '---')}")
    print(
        f"\tCache creation input tokens: {getattr(response.usage, 'cache_creation_input_tokens', '---')}"
    )

## 예제 1: 표준 Prompt Caching (Speculative Caching 미사용)

먼저 표준 prompt caching의 작동 방식을 살펴봅니다. 사용자가 질문을 입력한 다음 전체 context + 질문을 API에 보냅니다.

In [85]:
async def standard_prompt_caching_demo():
    client = AsyncAnthropic()

    # 대규모 context 준비
    initial_message = await create_initial_message()

    # 사용자 입력 시간 시뮬레이션(실제 app에서는 실제 사용자 입력)
    print("User is typing their question...")
    await asyncio.sleep(3)  # 3초 입력 시뮬레이션
    user_question = "What is the purpose of the BtShared structure?"
    print(f"User submitted: {user_question}")

    # 이제 전체 요청(context + 질문) 전송
    full_message = copy.deepcopy(initial_message)
    full_message["content"].append(
        {"type": "text", "text": f"Answer the user's question: {user_question}"}
    )

    print("\nSending request to API...")
    start_time = time.time()

    # 첫 token까지 시간 측정
    first_token_time = None
    async with client.messages.stream(
        messages=[full_message],
        model=MODEL,
        **DEFAULT_CLIENT_ARGS,
    ) as stream:
        async for text in stream.text_stream:
            if first_token_time is None and text.strip():
                first_token_time = time.time() - start_time
                print(f"\n🕐 Time to first token: {first_token_time:.2f} seconds")
                break

        # 전체 응답 얻기
        response = await stream.get_final_message()

    total_time = time.time() - start_time
    print(f"Total response time: {total_time:.2f} seconds")
    print_query_statistics(response, "Standard Caching")

    return first_token_time, total_time

In [86]:
# 표준 데모 실행
standard_ttft, standard_total = await standard_prompt_caching_demo()

Successfully downloaded btree.h
Successfully downloaded btree.c
Downloaded 2 files in 0.30 seconds
User is typing their question...
User submitted: What is the purpose of the BtShared structure?

Sending request to API...

🕐 Time to first token: 20.87 seconds
Total response time: 28.32 seconds

Standard Caching query statistics:
	Input tokens: 22
	Output tokens: 362
	Cache read input tokens: 0
	Cache creation input tokens: 151629


## 예제 2: Speculative Prompt Caching

이제 speculative prompt caching이 사용자의 입력 중 cache를 예열해 TTFT를 개선하는 모습을 살펴봅니다.

In [87]:
async def speculative_prompt_caching_demo():
    client = AsyncAnthropic()

    # 사용자는 상호작용하려는 대규모 context를 가지고 있음,
    # 이 경우 sqlite b-tree 구현(약 150k token).
    initial_message = await create_initial_message()

    # 사용자 입력 중 speculative caching 시작
    print("User is typing their question...")
    print("🔥 Starting cache warming in background...")

    # 사용자가 질문을 입력하는 동안 상호작용할 context에서 token 하나를
    # sampling하고, 명시적 prompt caching을 켜서
    # cache를 예열.
    cache_task = asyncio.create_task(sample_one_token(client, [initial_message]))

    # 사용자 입력 시간 시뮬레이션
    await asyncio.sleep(3)  # 3초 입력 시뮬레이션
    user_question = "What is the purpose of the BtShared structure?"
    print(f"User submitted: {user_question}")

    # cache 예열 완료 확인
    await cache_task
    print("✅ Cache warming completed!")

    # cached query용 message 준비. cache hit를 보장하기 위해
    # cache한 것과 동일한 초기 message를 재사용.
    cached_message = copy.deepcopy(initial_message)
    cached_message["content"].append(
        {"type": "text", "text": f"Answer the user's question: {user_question}"}
    )

    print("\nSending request to API (with warm cache)...")
    start_time = time.time()

    # 첫 token까지 시간 측정
    first_token_time = None
    async with client.messages.stream(
        messages=[cached_message],
        model=MODEL,
        **DEFAULT_CLIENT_ARGS,
    ) as stream:
        async for text in stream.text_stream:
            if first_token_time is None and text.strip():
                first_token_time = time.time() - start_time
                print(f"\n🚀 Time to first token: {first_token_time:.2f} seconds")
                break

        # 전체 응답 얻기
        response = await stream.get_final_message()

    total_time = time.time() - start_time
    print(f"Total response time: {total_time:.2f} seconds")
    print_query_statistics(response, "Speculative Caching")

    return first_token_time, total_time

In [88]:
# speculative caching 데모 실행
speculative_ttft, speculative_total = await speculative_prompt_caching_demo()

Successfully downloaded btree.h
Successfully downloaded btree.c
Downloaded 2 files in 0.36 seconds
User is typing their question...
🔥 Starting cache warming in background...
User submitted: What is the purpose of the BtShared structure?
✅ Cache warming completed!

Sending request to API (with warm cache)...

🚀 Time to first token: 1.94 seconds
Total response time: 8.40 seconds

Speculative Caching query statistics:
	Input tokens: 22
	Output tokens: 330
	Cache read input tokens: 151629
	Cache creation input tokens: 0


## 성능 비교

결과를 비교하여 speculative caching의 이점을 살펴봅니다.

In [89]:
print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)

print("\nStandard Prompt Caching:")
print(f"  Time to First Token: {standard_ttft:.2f} seconds")
print(f"  Total Response Time: {standard_total:.2f} seconds")

print("\nSpeculative Prompt Caching:")
print(f"  Time to First Token: {speculative_ttft:.2f} seconds")
print(f"  Total Response Time: {speculative_total:.2f} seconds")

ttft_improvement = (standard_ttft - speculative_ttft) / standard_ttft * 100
total_improvement = (standard_total - speculative_total) / standard_total * 100

print("\n🎯 IMPROVEMENTS:")
print(
    f"  TTFT Improvement: {ttft_improvement:.1f}% ({standard_ttft - speculative_ttft:.2f}s faster)"
)
print(
    f"  Total Time Improvement: {total_improvement:.1f}% ({standard_total - speculative_total:.2f}s faster)"
)

PERFORMANCE COMPARISON

Standard Prompt Caching:
  Time to First Token: 20.87 seconds
  Total Response Time: 28.32 seconds

Speculative Prompt Caching:
  Time to First Token: 1.94 seconds
  Total Response Time: 8.40 seconds

🎯 IMPROVEMENTS:
  TTFT Improvement: 90.7% (18.93s faster)
  Total Time Improvement: 70.4% (19.92s faster)


## 핵심 요점

1. **Speculative caching은 TTFT를 크게 줄입니다**, 사용자가 입력하는 동안 cache를 예열하기 때문입니다.
2. **이 패턴은 가장 효과적입니다**, 여러 query에서 재사용되는 큰 context(>1000 token)에서 그렇습니다.
3. **구현은 간단합니다** - 사용자가 입력하는 동안 1-token 요청을 보내면 됩니다.
4. **Cache 예열은 병렬로 이루어집니다**, 사용자 입력과 동시에 진행되어 cache 생성 시간을 사실상 "숨깁니다".

## 모범 사례

- 가능한 한 일찍 cache 예열을 시작하세요(예: 사용자가 입력 field에 focus할 때).
- cache hit를 보장하도록 예열과 실제 요청에 정확히 같은 context를 사용하세요.
- `cache_read_input_tokens`를 모니터링해 cache hit를 확인하세요.
- session 간 원치 않는 cache 공유를 막기 위해 timestamp를 추가하세요.